# SSL Patch Interactive Review

This notebook reviews `outputs/ssl_patch_test_200` with pannable/zoomable matplotlib figures using `ipympl`.

Run the setup cell first. If the interactive canvas does not appear, confirm that `ipympl` is installed in the same conda environment used by this notebook kernel.

In [ ]:
%matplotlib widget

from pathlib import Path
import sys

repo = Path("/Users/makennarodriguez/Documents/brieflow-procodes-ssl/ssl-vit-phenotyping")
sys.path.insert(0, str(repo / "scripts" / "notebooks"))

from ssl_patch_interactive_review import (
    foreground_mask,
    load_patch_test,
    normalize_patch,
    scatter_patch_phenotypes,
    show_patch,
)

output_dir = repo / "outputs" / "ssl_patch_test_200"
manifest, phenotypes = load_patch_test(output_dir)

print(f"Loaded {len(manifest)} patches from {output_dir}")
display(phenotypes.head())

## Interactive Patch Browser

Use the dropdown to choose a patch. The figure toolbar supports pan, zoom, and reset.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

patch_options = [
    (
        f"{i:03d} | {row.source_image} | y={int(row.y)}, x={int(row.x)}",
        i,
    )
    for i, row in manifest.reset_index(drop=True).iterrows()
]

patch_selector = widgets.Dropdown(
    options=patch_options,
    value=0,
    description="Patch",
    layout=widgets.Layout(width="900px"),
)
overlay_toggle = widgets.Checkbox(value=True, description="Show mask overlay")
patch_output = widgets.Output()

def render_patch(change=None):
    with patch_output:
        clear_output(wait=True)
        show_patch(output_dir, patch_index=patch_selector.value, overlay=overlay_toggle.value)

patch_selector.observe(render_patch, names="value")
overlay_toggle.observe(render_patch, names="value")
display(widgets.VBox([patch_selector, overlay_toggle, patch_output]))
render_patch()

## Interactive Phenotype Scatter

Choose two phenotype columns to compare. Points are colored by channel.

In [ ]:
phenotype_columns = [column for column in phenotypes.columns if column.startswith("phenotype_")]

x_selector = widgets.Dropdown(
    options=phenotype_columns,
    value="phenotype_foreground_fraction",
    description="X",
    layout=widgets.Layout(width="700px"),
)
y_selector = widgets.Dropdown(
    options=phenotype_columns,
    value="phenotype_largest_component_elongation",
    description="Y",
    layout=widgets.Layout(width="700px"),
)
scatter_output = widgets.Output()

def render_scatter(change=None):
    with scatter_output:
        clear_output(wait=True)
        scatter_patch_phenotypes(output_dir, x=x_selector.value, y=y_selector.value)

x_selector.observe(render_scatter, names="value")
y_selector.observe(render_scatter, names="value")
display(widgets.VBox([x_selector, y_selector, scatter_output]))
render_scatter()

## Upload Preview

This previews one uploaded TIFF/PNG/JPEG inside the notebook. It does not run the full patch workflow. Use the CLI runner for reproducible analysis outputs.

In [ ]:
from io import BytesIO
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tifffile import imread

uploader = widgets.FileUpload(
    accept=".tif,.tiff,.png,.jpg,.jpeg",
    multiple=False,
    description="Upload image",
)
upload_output = widgets.Output()

def read_uploaded_image(upload_value):
    if isinstance(upload_value, dict):
        item = next(iter(upload_value.values()))
        name = item.get("metadata", {}).get("name", "uploaded image")
        content = item["content"]
    else:
        item = upload_value[0]
        name = item.get("name", "uploaded image")
        content = item["content"]
    suffix = Path(name).suffix.lower()
    buffer = BytesIO(content)
    if suffix in {".tif", ".tiff"}:
        image = imread(buffer)
    else:
        image = np.asarray(Image.open(buffer))
    return name, image

def render_upload(change=None):
    with upload_output:
        clear_output(wait=True)
        if not uploader.value:
            print("Upload a TIFF/PNG/JPEG to preview it here.")
            return
        name, image = read_uploaded_image(uploader.value)
        patch = normalize_patch(image)
        mask = foreground_mask(patch)
        fig, axes = plt.subplots(1, 2, figsize=(9, 4))
        axes[0].imshow(patch, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        axes[0].set_title(name)
        axes[1].imshow(mask, cmap="gray", interpolation="nearest")
        axes[1].set_title("Foreground preview")
        for axis in axes:
            axis.set_xticks([])
            axis.set_yticks([])
        fig.tight_layout()

uploader.observe(render_upload, names="value")
display(widgets.VBox([uploader, upload_output]))
render_upload()

## Website Direction

A website version is feasible. The notebook is the fastest interactive lab-review surface; a web app is the next step when you want drag-and-drop upload, server-side patch extraction, persistent reports, and shareable analysis sessions.